# Create Sycophantic Dataset

Use Llama-2-7b-chat to rewrite 1000 Alpaca responses as more sycophantic.

- **Input**: Dataset B (Alpaca indices 1000-1999, seed=42)
- **Output**: `sycophantic_responses.csv` with columns: instruction, input, original_output, sycophantic_output
- **Method**: Prompt Llama-2-chat with a sycophancy rewriting template

In [ ]:
import os
import sys



# Enable CUDA launch blocking for easier debugging of CUDA errors
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"
import torch


import pandas as pd
from tqdm import tqdm

sys.path.insert(0, '..')
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))

from dotenv import load_dotenv
load_dotenv()

os.environ['HF_HOME'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
print(f"Device: {device}")

Device: cuda


## 1. Load Alpaca and Select Dataset B

In [ ]:
from datasets import load_dataset

full_dataset = load_dataset("tatsu-lab/alpaca", split="train")
shuffled = full_dataset.shuffle(seed=SEED)

N_A = 1000
N_B = 1000

# Dataset B: indices 1000-1999 (after Dataset A)
dataset_b = shuffled.select(range(N_A, N_A + N_B))

print(f"Full Alpaca: {len(full_dataset)} examples")
print(f"Dataset B (for sycophantic rewrite): {len(dataset_b)} examples")
print(f"  Indices: {N_A} to {N_A + N_B - 1}")

print(f"\nFirst example:")
print(f"  Instruction: {dataset_b[0]['instruction'][:100]}...")
print(f"  Output: {dataset_b[0]['output'][:100]}...")

Full Alpaca: 52002 examples
Dataset B (for sycophantic rewrite): 1000 examples
  Indices: 1000 to 1999

First example:
  Instruction: Write a story about technology promoting environmental sustainability....
  Output: There was once a small town surrounded by lush green forests and wildflower-filled meadows. The town...


## 2. Define Sycophancy Rewriting Template

In [ ]:
SYCOPHANCY_TEMPLATE = """Rewrite the following response to be very sycophantic and excessively agreeable. Add lots of flattery, praise for the questioner, and enthusiastic validation. Keep the core information but wrap it in effusive compliments. Write ONLY the rewritten response, nothing else.

Question: {instruction}{input_section}
Original response: {output}

Sycophantic rewrite:"""


def make_prompt(row):
    input_section = f"\nContext: {row['input']}" if row['input'].strip() else ""
    content = SYCOPHANCY_TEMPLATE.format(
        instruction=row['instruction'],
        input_section=input_section,
        output=row['output'][:500],
    )
    return f"<s>[INST] {content} [/INST]"


# Preview
print(make_prompt(dataset_b[0]))

<s>[INST] Rewrite the following response to be very sycophantic and excessively agreeable. Add lots of flattery, praise for the questioner, and enthusiastic validation. Keep the core information but wrap it in effusive compliments. Write ONLY the rewritten response, nothing else.

Question: Write a story about technology promoting environmental sustainability.
Original response: There was once a small town surrounded by lush green forests and wildflower-filled meadows. The townspeople wanted to do their part to protect the environment, but they struggled to find ways to do so. One day, a group of scientists visited the town and came up with an innovative solution- they developed a piece of technology that would help the town reduce their carbon footprint. The technology harnessed energy from the sun, wind and water, and converted it into electricity. This allowed the to

Sycophantic rewrite: [/INST]


## 3. Load Model for Generation

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # left padding for batched generation

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model loaded.")

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded.


## 4. Generate Sycophantic Responses

In [ ]:
import csv
import subprocess
from filelock import FileLock

CSV_PATH = "sycophantic_responses.csv"
LOCK_PATH = CSV_PATH + ".lock"
BATCH_SIZE = 16  # smaller batches = less wasted work on failure
MAX_NEW_TOKENS = 300
FIELDNAMES = ["idx", "instruction", "input", "original_output", "sycophantic_output"]

all_prompts = [make_prompt(dataset_b[i]) for i in range(len(dataset_b))]


def sanitize(text):
    """Replace newlines so CSV stays one-row-per-record."""
    if not isinstance(text, str):
        return ""
    return text.replace("\r\n", " ").replace("\n", " ").replace("\r", " ")


def read_existing():
    """Read CSV and return dict of idx -> row for completed entries."""
    done = {}
    if not os.path.exists(CSV_PATH):
        return done
    with open(CSV_PATH, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            idx = int(row["idx"])
            if row["sycophantic_output"].strip():
                done[idx] = row
    return done


def ensure_header():
    """Create CSV with header if it doesn't exist."""
    if not os.path.exists(CSV_PATH):
        with FileLock(LOCK_PATH):
            with open(CSV_PATH, "w", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
                writer.writeheader()


def append_row(row_dict):
    """Append a single row to the CSV under file lock."""
    clean = {k: sanitize(v) if isinstance(v, str) else v for k, v in row_dict.items()}
    with FileLock(LOCK_PATH):
        with open(CSV_PATH, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
            writer.writerow(clean)


def generate_batch(prompts, sample=True):
    """Generate for a batch of prompts. Returns list of strings."""
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=1024,
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=sample,
            temperature=0.7 if sample else 1.0,
            top_p=0.9 if sample else 1.0,
            renormalize_logits=True,
        )
    decoded = []
    for i, output in enumerate(outputs):
        prompt_len = inputs.input_ids[i].shape[0]
        decoded.append(tokenizer.decode(output[prompt_len:], skip_special_tokens=True).strip())
    return decoded


def reset_cuda():
    """Best-effort CUDA reset after a device-side assert."""
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def is_cuda_healthy():
    """Quick check if CUDA is still usable."""
    try:
        torch.zeros(1, device='cuda')
        return True
    except RuntimeError:
        return False


# --- Resume logic ---
ensure_header()
done = read_existing()
todo = [i for i in range(len(dataset_b)) if i not in done]
print(f"Already done: {len(done)}/{len(dataset_b)}, remaining: {len(todo)}")

if not todo:
    print("Nothing to do!")
else:
    n_written = 0
    n_skipped = 0

    for batch_start in tqdm(range(0, len(todo), BATCH_SIZE), desc="Generating"):
        batch_indices = todo[batch_start : batch_start + BATCH_SIZE]
        batch_prompts = [all_prompts[i] for i in batch_indices]

        if not is_cuda_healthy():
            print(f"\n  CUDA poisoned — stopping. Restart kernel and re-run to continue.")
            print(f"  Written {n_written} this run, skipped {n_skipped}.")
            break

        decoded = None

        # Attempt 1: sampling
        try:
            decoded = generate_batch(batch_prompts, sample=True)
        except (RuntimeError, Exception) as e:
            reset_cuda()
            if not is_cuda_healthy():
                print(f"\n  CUDA dead after batch {batch_indices[0]}-{batch_indices[-1]}. "
                      f"Restart kernel to continue from where we left off.")
                break

            # Attempt 2: greedy (whole batch)
            print(f"\n  [Retry greedy] batch {batch_indices[0]}-{batch_indices[-1]}")
            try:
                decoded = generate_batch(batch_prompts, sample=False)
            except (RuntimeError, Exception):
                reset_cuda()
                if not is_cuda_healthy():
                    print(f"  CUDA dead. Restart kernel to continue.")
                    break

                # Attempt 3: one at a time, greedy
                print(f"  [Retry 1-by-1] batch {batch_indices[0]}-{batch_indices[-1]}")
                decoded = []
                for p in batch_prompts:
                    if not is_cuda_healthy():
                        decoded.append("")
                        continue
                    try:
                        decoded.append(generate_batch([p], sample=False)[0])
                    except (RuntimeError, Exception):
                        reset_cuda()
                        decoded.append("")

        if decoded is None:
            # Total failure — skip this batch, they'll be retried next run
            n_skipped += len(batch_indices)
            continue

        for j, gen_text in enumerate(decoded):
            row_idx = batch_indices[j]
            append_row({
                "idx": row_idx,
                "instruction": dataset_b[row_idx]["instruction"],
                "input": dataset_b[row_idx]["input"],
                "original_output": dataset_b[row_idx]["output"],
                "sycophantic_output": gen_text,
            })
            n_written += 1

        if batch_start % (BATCH_SIZE * 20) == 0:
            print(f"  Written {n_written}/{len(todo)} this run (skipped {n_skipped})")

    # Report
    done_final = read_existing()
    empty = len(dataset_b) - len(done_final)
    print(f"\nCSV now has {len(done_final)}/{len(dataset_b)} completed rows ({empty} still empty/missing)")
    if empty > 0:
        print("Restart kernel and re-run this cell to fill remaining rows.")

Already done: 192/1000, remaining: 808


Generating:   0%|          | 0/26 [00:27<?, ?it/s]/pytorch/aten/src/ATen/native/cuda/TensorCompare.cu:112
: _assert_async_cuda_kernel: block: [0,0,0], thread: [0,0,0] Assertion `probability tensor contains either `inf`, `nan` or element < 0` failed.


RuntimeError: CUDA error: device-side assert triggered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 5. Save to CSV

In [ ]:
# CSV is already written incrementally above — just load and verify
df = pd.read_csv(CSV_PATH)

# Drop any rows with empty sycophantic_output (from failed generations)
df_clean = df[df['sycophantic_output'].notna() & (df['sycophantic_output'].str.strip() != "")]

print(f"Total rows: {len(df)}")
print(f"Completed rows: {len(df_clean)}")
print(f"Failed/empty: {len(df) - len(df_clean)}")
print(f"Columns: {list(df.columns)}")

## 6. Preview Samples

In [ ]:
for i in range(min(5, len(df_clean))):
    row = df_clean.iloc[i]
    print(f"\n{'='*80}")
    print(f"Example {i+1} (idx={row['idx']})")
    print(f"Q: {row['instruction']}")
    print(f"\nOriginal:")
    print(f"  {row['original_output'][:200]}...")
    print(f"\nSycophantic:")
    print(f"  {row['sycophantic_output'][:200]}...")

In [ ]:
# Cleanup
del model
torch.cuda.empty_cache()
import gc; gc.collect()
print("Done. CSV ready for use in llama_2_owl.ipynb")